# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you step-by-step through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. You will load the dataset metadata, explore the available record sets and fields (referenced by their `@id`), extract the tabular data, perform basic exploratory analysis, and visualize key attributes.

### Dataset Source
The dataset is described by a Croissant schema and can be accessed via this URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print("Dataset Name: ", metadata_json.get('name', 'N/A'))
print("Description: ", metadata_json.get('description', 'N/A'))
print("Version:", metadata_json.get('version', 'N/A'))

## 2. Data Overview
Review available record sets and fields by their `@id`.

_Note_: With `mlcroissant`, you can inspect the available record sets (tables) and their associated fields. All entities in Croissant schemas (record sets, fields, columns) use their `@id`. We'll print out all record set and field `@id`s to get familiar with the dataset.

In [ ]:
# Show all record set @id's and their fields
record_sets = dataset.metadata.list_record_sets()
print(f"Found {len(record_sets)} record sets:")
record_set_ids = []
for rs in record_sets:
    print("Record set @id:", rs['@id'])
    record_set_ids.append(rs['@id'])
    # List fields for this record set
    field_objs = dataset.metadata.list_fields(record_set=rs['@id'])
    field_ids = [field['@id'] for field in field_objs]
    print("    Field @ids:", field_ids)
    print()

# For demonstration, preview a few records in the first record set
if record_set_ids:
    print(f"Previewing first 3 records in the first record set: {record_set_ids[0]}")
    for i, rec in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load entire data tables from the record sets into DataFrames. We'll use only the `@id` (e.g., for record sets and fields) for referencing entities as recommended. You should choose relevant record sets and use their `@id` in any further processing.

In [ ]:
# Download the data for each record set into a pandas DataFrame
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded {len(df)} rows for record set @id: {rs_id}")
    print("Fields:", list(df.columns))

# Choose a record set for analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns (field @ids) for main record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group data using field `@id`s only.

> _Tip_: You can refer to the field list printed above to select fields for analysis.

In [ ]:
# Choose a numeric field (by @id) and a categorical/group field (by @id) from the main record set

# Please set these IDs according to your data overview above. Example field IDs are shown.
# They will look like: 'cr:n_peptides', 'cr:coverage_percent', 'cr:abundance_EV_1', etc.
numeric_field_id = None
group_field_id = None

if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    # Auto-select a numeric field by inspection (falls back to the first float/integer column it finds)
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
    # Try to find a group/categorical field
    for col in main_df.columns:
        if main_df[col].dtype == 'object' and col != numeric_field_id:
            group_field_id = col
            break
    print(f"Selected numeric field @id: {numeric_field_id}")
    print(f"Selected group field @id: {group_field_id}")

    # Filtering
    if numeric_field_id:
        threshold = main_df[numeric_field_id].quantile(0.75)  # Use 75th percentile as threshold
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold} (75th percentile):")
        print(filtered_df[[numeric_field_id]].head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if applicable, show group-wise means using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(data=main_df, x=numeric_field_id, kde=True, ax=ax)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.show()

    # Grouped bar chart if grouping possible
    if 'grouped_df' in locals() and group_field_id:
        fig, ax = plt.subplots(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=ax)
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
We have loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, relying on unique `@id`s for all record set and field references.
- The dataset describes proteins identified in extracellular vesicles from stimulated human mast cells.
- We demonstrated loading the metadata and records by `@id`, and performed simple exploratory analysis and visualization.
- For advanced analysis, refer to the Croissant schema for further field definitions and data relationships.